# I/O Utilities
> Read and write configuration files (JSON, TOML)

In [ ]:
#| default_exp utils.io

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
import json
import os
import subprocess
import platform
from pathlib import Path
from typing import Optional

## JSON Configuration

In [ ]:
#| export
def read_json_config(path: Path) -> dict:
    """Read a JSON configuration file.
    
    Returns empty dict if file doesn't exist or is invalid.
    
    Parameters
    ----------
    path : Path
        Path to JSON file.
    
    Returns
    -------
    dict
        Parsed JSON or empty dict on error.
    """
    if not path.exists():
        return {}
    try:
        content = path.read_text().strip()
        return json.loads(content) if content else {}
    except (json.JSONDecodeError, OSError):
        return {}

In [ ]:
#| export
def write_json_config(path: Path, config: dict, indent: int = 2) -> None:
    """Write a JSON configuration file.
    
    Creates parent directories if needed.
    
    Parameters
    ----------
    path : Path
        Path to JSON file.
    config : dict
        Configuration to write.
    indent : int
        JSON indentation level.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(config, indent=indent))

## TOML Configuration

In [ ]:
#| export
def read_toml_config(path: Path) -> dict:
    """Read a TOML configuration file.
    
    Returns empty dict if file doesn't exist or is invalid.
    Uses tomllib (Python 3.11+) or tomli as fallback.
    
    Parameters
    ----------
    path : Path
        Path to TOML file.
    
    Returns
    -------
    dict
        Parsed TOML or empty dict on error.
    """
    if not path.exists():
        return {}
    try:
        import tomllib
        return tomllib.loads(path.read_text())
    except ImportError:
        try:
            import tomli
            return tomli.loads(path.read_text())
        except ImportError:
            return {}
    except Exception:
        return {}

In [ ]:
#| export
def write_toml_config(path: Path, config: dict) -> None:
    """Write a TOML configuration file.
    
    Creates parent directories if needed.
    Uses tomli_w if available, otherwise manual formatting.
    
    Parameters
    ----------
    path : Path
        Path to TOML file.
    config : dict
        Configuration to write.
    """
    try:
        import tomli_w
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(tomli_w.dumps(config))
    except ImportError:
        # Manual TOML generation
        path.parent.mkdir(parents=True, exist_ok=True)
        lines = []
        for key, value in config.items():
            if isinstance(value, dict):
                for sub_key, sub_value in value.items():
                    lines.append(f'[{key}.{sub_key}]')
                    for k, v in sub_value.items():
                        if isinstance(v, list):
                            args_str = ', '.join(f'"{a}"' for a in v)
                            lines.append(f'{k} = [{args_str}]')
                        elif isinstance(v, str):
                            lines.append(f'{k} = "{v}"')
                        else:
                            lines.append(f'{k} = {v}')
                    lines.append('')
            elif isinstance(value, str):
                lines.append(f'{key} = "{value}"')
            else:
                lines.append(f'{key} = {value}')
        path.write_text('\n'.join(lines))

## File Operations

In [ ]:
#| export
def open_file_in_editor(path: Path) -> bool:
    """Open a file in the default system editor.
    
    Parameters
    ----------
    path : Path
        Path to file to open.
    
    Returns
    -------
    bool
        True if successful.
    """
    if not path.exists():
        return False
    
    system = platform.system().lower()
    try:
        if system == 'darwin':
            subprocess.run(['open', str(path)], check=True)
        elif system == 'windows':
            os.startfile(str(path))
        else:
            subprocess.run(['xdg-open', str(path)], check=True)
        return True
    except Exception:
        return False

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()